# ASFA: Artifact-First Submission Forensics Audit

**Companion notebook to _The ZIP Is the Treatment_**  
Author: Chukka Venugopalam  

This notebook demonstrates a reproducible, artifact-first audit of scored software-engineering-agent submissions. It asks whether scored ZIPs actually contain different payloads, computes the case-study's descriptive summaries, and demonstrates canonical archive fingerprinting.

**Scope:** this is an audit notebook, not a model runner or benchmark evaluator. It contains no model weights, private tasks, repository snapshots, hidden tests, or execution traces. The 58-task mapping is a participant-derived working assumption, not an official denominator disclosure. The observed leaderboard scores remain aggregate outcomes.

The notebook uses only Python's standard library and runs on CPU without internet access.

In [1]:
import csv, io, itertools, json, math, statistics, hashlib, zipfile, tempfile
from pathlib import Path

# The source tables are embedded so the notebook is self-contained.
ARCHIVE_CSV = r'''archive,displayed_score,solves_if_n58,archive_sha256,content_fingerprint,root_tools,graph_tools,sub_agents,skills,adapters,temperature,top_k,max_output_tokens,thinking_budget,max_tool_calls,max_time_minutes,max_turns,timeout_seconds,prompt_bytes
score-0.00.zip,0.0,0,fb4780f35409c97581df5b67a9e5835439e59d82afd04f8903b6c35d5a797895,c067d5c16047df009d0e959fd925696e7c9e36b2d7cd7ee71c74c63bbeb143e0,11,1,2,1,0,0.15,,16384,4096,100,60,500,300,8404
score-0.03.zip,0.03,2,359a50986e9c23926cd99da853ad30fa399291947a9e9fc68b5906339156cf1b,73a18d30860455353558dada225fae0c10a4c68e3f6d13e1a013422886d86fbd,6,0,0,0,0,0.2,40,3072,,50,5,100,300,3043
score-0.06.zip,0.06,4,dcbcafff7b54210efd842a68b0790f735c51e4f8504ffdb4774d08b4cd59522f,ca149af8818b9101bc0e16bdc594b003ff7b51dffd54baaf3b54983329e1,9,1,0,1,0,0.2,40,4096,,50,4.5,80,300,8637
score-0.06(!).zip,0.06,4,cb9e2ae63a5c62cd53fec8221541b0f21fe52a541fcc035fa9fb92d634161cef,c711d018189332fa372cd52ecd7b19a1c3e52cf916189fb886ffa62bf1edd8fb,7,0,1,0,0,0.2,40,2048,,50,5,80,300,3816
score-0.12.zip,0.12,7,e15b4cbcb48e6e66be625b840958fc27503a66598904dc66a2b3b4958df30a80,d47219b5b31d1b42b5f02952398e73618c8331612039cd69d29c7710b0f2c8ca,9,1,0,0,0,0.2,40,8192,4096,28,8,80,240,4799
score-0.12(1).zip,0.12,7,52b683e1224d942abc563b94b460544112abb488ce71e184c2d6d66bdffa76ef,9ec934d798131d7dff5fa1a7abbb1aa0ccb37ee967b1f29549fa063d1050023a,6,0,0,0,0,1.0,64,8192,4096,28,8,80,240,4592
score-0.12(2).zip,0.12,7,5b58fc402eb065904fb294fc4341ec6e2a35a036935d237976341da323e302e6,1071413149f88932445fd80fccac21d2e6af26509d789552c53157253a025096,6,0,0,0,0,1.0,64,8192,4096,28,8,80,240,4413
score-0.13.zip,0.13,8,d9b913d6a401582dd07f3c16e08ff1adb9e7869255478a8c01366bb2c9aba454,e293d702bb35a86c6e122bd1e2c3fb493906f74c199698bc4ab82f9cb361228d,6,0,0,0,0,0.2,40,8192,4096,28,8,80,240,4052
score-0.13(1).zip,0.13,8,df8e4f5876d19e1692e4eca2c8dbe211bf61ca67f46d05a33f1023676893aa1e,bd8718624dd8e54de43e04d052b642dd551719fdc8c8574b64afb5e1730bd19b,10,1,1,0,2,0.2,,16384,4096,28,8,80,240,3240
score-0.15.zip,0.15,9,46491726209417324f7c4e99894331e1f559af67c7e6cd1eaad8a0afa951ceab,abe6f1018e788f64e09cf26288f33c4d60dafd011690cf11541836dc0cc9ebaa,6,0,0,0,0,1.0,64,8192,4096,28,8,80,240,3240
score-0.17.zip,0.17,10,be1aa4472ac81dcf801d459743d2ec774a5e03153959129ba96913eec0895747,abe6f1018e788f64e09cf26288f33c4d60dafd011690cf11541836dc0cc9ebaa,6,0,0,0,0,1.0,64,8192,4096,28,8,80,240,3240
score-0.18.zip,0.18,11,dd1e015ca058c7680edc0a7115eac2d87d041d6bc2cc5cd0f8bc27081a625f26,52662ceeec309cbbfeb1d51beb462af05324d62d21d7de1d0982149097104e9c,9,1,0,0,0,0.2,40,8192,4096,28,8,100,300,4571
score-0.18(1).zip,0.18,11,2394bbdec0fcd0700effb5f4f79de3f338f3fd4a8df5c1f1d5882da2d54457fa,901adfb40d78b8b37c2001b0aeedcc4d9fc86d54f3baf854435d85e9bbb7a595,9,1,0,0,0,0.2,40,8192,4096,28,8,100,300,4019
score-0.20.zip,0.2,12,be1aa4472ac81dcf801d459743d2ec774a5e03153959129ba96913eec0895747,abe6f1018e788f64e09cf26288f33c4d60dafd011690cf11541836dc0cc9ebaa,6,0,0,0,0,1.0,64,8192,4096,28,8,80,240,3240'''
FEATURE_CSV = r'''payload_group,archive_labels,solves_mean,score_records,graph_tools_declared,temperature_0_2,max_turns_100_timeout_300,has_sub_agent,prompt_over_4KB,hard_edit_by_12,soft_edit_by_12,anti_repetition_rules,status_90s_rule,one_check_then_stop,never_bare_pytest,evidence_ledger,test_obligations,call_checkpoints_20_24,reproduce_step,graph_tool_guidance
5dff4c611a,score-0.12.zip,7,1,1,1,0,0,1,0,1,1,0,0,0,0,0,0,1,1
cc9391d93c,score-0.12(1).zip,7,1,0,0,0,0,1,0,0,0,1,0,1,1,0,0,1,0
648d753cca,score-0.12(2).zip,7,1,0,0,0,0,1,1,0,0,1,1,1,0,1,0,0,0
eccec14fdf,score-0.13.zip,8,1,0,1,0,0,1,0,1,1,0,1,0,0,0,0,0,0
ce10b18cc8,score-0.13(1).zip,8,1,1,1,0,1,0,1,0,0,1,1,1,0,0,0,0,0
4cb9c36a16,score-0.15+0.17+0.20,10.3333333333,3,0,0,0,0,0,1,0,0,1,1,1,0,0,0,0,0
18c77ac6dd,score-0.18.zip,11,1,1,1,1,0,1,0,0,1,0,0,0,0,0,0,1,1
810e018e21,score-0.18(1).zip,11,1,1,1,1,0,1,0,0,1,0,0,0,0,0,1,1,1'''
FOLLOWUP_CSV = r'''experiment,change,baseline_value,candidate_value,observed_score,score_zip_sha256,repeats,interpretation
I02,max_output_tokens,8192,16384,0.12,42c9e301ad05b8c06aaf93eb17872302f2fbba4b8bcc888c4182d9f0fc493003,1,single public-score draw; no demonstrated benefit
B2,thinking_budget,4096,2048,0.15,4e1f0f10af06f3253da571cd2af90cae681a15ae5629c73b2ad5092391378d08,1,single public-score draw; no demonstrated benefit'''

def rows(text):
    return list(csv.DictReader(io.StringIO(text)))

archives = rows(ARCHIVE_CSV)
features = rows(FEATURE_CSV)
followups = rows(FOLLOWUP_CSV)
print(f"Loaded {len(archives)} scored archive records, {len(features)} distinct in-envelope payload rows, and {len(followups)} follow-up trials.")


Loaded 14 scored archive records, 8 distinct in-envelope payload rows, and 2 follow-up trials.


## 1. Recompute the headline counts

The archive SHA-256 identifies the exact ZIP bytes. The `content_fingerprint` identifies the extracted file payload independently of ZIP timestamps and compression metadata. Multiple ZIP records can therefore represent one payload.

In [2]:
fingerprints = {r['content_fingerprint'] for r in archives}
baseline_fp = 'abe6f1018e788f64e09cf26288f33c4d60dafd011690cf11541836dc0cc9ebaa'
baseline = sorted([r for r in archives if r['content_fingerprint'] == baseline_fp], key=lambda r: float(r['displayed_score']))

assert len(archives) == 14
assert len(fingerprints) == 12
assert [float(r['displayed_score']) for r in baseline] == [0.15, 0.17, 0.20]

print('Archive records:', len(archives))
print('Distinct extracted payloads:', len(fingerprints))
print('Same baseline payload scores:', [float(r['displayed_score']) for r in baseline])
print('Same baseline solves under the 58-task assumption:', [int(r['solves_if_n58']) for r in baseline])


Archive records: 14
Distinct extracted payloads: 12
Same baseline payload scores: [0.15, 0.17, 0.2]
Same baseline solves under the 58-task assumption: [9, 10, 12]


## 2. Recompute the descriptive execution-envelope comparison

The core envelope is defined by `max_tool_calls=28`, `max_time_minutes=8`, and `thinking_budget=4096`. This definition deliberately does **not** require one output-token setting: nine of the ten records used 8,192 tokens and one adapter-augmented record used 16,384. The comparison is observational and confounded; it is not a causal estimate.

In [3]:
core = [r for r in archives if (r['max_tool_calls'], r['max_time_minutes'], r['thinking_budget']) == ('28','8','4096')]
outside = [r for r in archives if r not in core]
core_solves = [int(r['solves_if_n58']) for r in core]
outside_solves = [int(r['solves_if_n58']) for r in outside]
output_caps = {cap: sum(r['max_output_tokens'] == cap for r in core) for cap in sorted({r['max_output_tokens'] for r in core})}

print('Core records:', len(core), 'mean solves:', round(statistics.mean(core_solves), 2), 'SD:', round(statistics.stdev(core_solves), 2))
print('Core output-token counts:', output_caps)
print('Outside-core records:', len(outside), 'solve counts:', outside_solves, 'mean:', round(statistics.mean(outside_solves), 2))
assert len(core) == 10 and len(outside) == 4
assert output_caps == {'16384': 1, '8192': 9}


Core records: 10 mean solves: 9 SD: 1.89
Core output-token counts: {'16384': 1, '8192': 9}
Outside-core records: 4 solve counts: [0, 2, 4, 4] mean: 2.5


## 3. Recompute the 16 exploratory factor contrasts

The analysis uses eight distinct payload-level means, not 10 archive records, to avoid treating repeated copies of the same baseline payload as independent designs. The interval is a heuristic normal interval using the 10-record core-envelope run standard deviation as a plug-in noise scale. It is **not** a calibrated small-sample confidence interval. The exact permutation p-values are nominal, exploratory, and unadjusted for 16 comparisons.

In [4]:
def mean(xs):
    return statistics.mean(xs) if xs else float('nan')

def exact_two_sided_permutation_p(values, flags):
    yes = [v for v, f in zip(values, flags) if f]
    no = [v for v, f in zip(values, flags) if not f]
    if not yes or not no:
        return None
    observed = abs(mean(yes) - mean(no))
    k, n = len(yes), len(values)
    total = extreme = 0
    for idx in itertools.combinations(range(n), k):
        chosen = set(idx)
        a = [values[i] for i in idx]
        b = [values[i] for i in range(n) if i not in chosen]
        total += 1
        if abs(mean(a) - mean(b)) >= observed - 1e-12:
            extreme += 1
    return extreme / total

values = [float(r['solves_mean']) for r in features]
sd_run = statistics.stdev(core_solves)
metadata = {'payload_group','archive_labels','solves_mean','score_records'}
results = []
for key in features[0]:
    if key in metadata:
        continue
    flags = [r[key] == '1' for r in features]
    yes = [v for v, f in zip(values, flags) if f]
    no = [v for v, f in zip(values, flags) if not f]
    if not yes or not no:
        continue
    diff = mean(yes) - mean(no)
    half_width = 1.96 * sd_run * math.sqrt(1/len(yes) + 1/len(no))
    p = exact_two_sided_permutation_p(values, flags)
    results.append((key, len(yes), len(no), round(mean(yes),2), round(mean(no),2), round(diff,2), (round(diff-half_width,2), round(diff+half_width,2)), round(p,3)))

for r in results:
    print(f"{r[0]:30s} n={r[1]}/{r[2]} with={r[3]:5.2f} without={r[4]:5.2f} diff={r[5]:+5.2f} heuristic_interval={r[6]} p={r[7]}")
assert len(results) == 16
assert sum(1 for r in results if r[6][0] <= 0 <= r[6][1]) == 15
assert next(r for r in results if r[0] == 'max_turns_100_timeout_300')[5:] == (3.11, (0.09, 6.13), 0.036)


graph_tools_declared           n=4/4 with= 9.25 without= 8.08 diff=+1.17 heuristic_interval=(-1.45, 3.78) p=0.4
temperature_0_2                n=5/3 with= 9.00 without= 8.11 diff=+0.89 heuristic_interval=(-1.81, 3.59) p=0.518
max_turns_100_timeout_300      n=2/6 with=11.00 without= 7.89 diff=+3.11 heuristic_interval=(0.09, 6.13) p=0.036
has_sub_agent                  n=1/7 with= 8.00 without= 8.76 diff=-0.76 heuristic_interval=(-4.71, 3.19) p=1.0
prompt_over_4KB                n=6/2 with= 8.50 without= 9.17 diff=-0.67 heuristic_interval=(-3.68, 2.35) p=0.679
hard_edit_by_12                n=3/5 with= 8.44 without= 8.80 diff=-0.36 heuristic_interval=(-3.05, 2.34) p=0.768
soft_edit_by_12                n=2/6 with= 7.50 without= 9.06 diff=-1.56 heuristic_interval=(-4.57, 1.46) p=0.429
anti_repetition_rules          n=4/4 with= 9.25 without= 8.08 diff=+1.17 heuristic_interval=(-1.45, 3.78) p=0.4
status_90s_rule                n=4/4 with= 8.08 without= 9.25 diff=-1.17 heuristic_interval=(-3

## 4. Follow-up single-factor trials

The output cap was doubled and the thinking budget was halved in separate submissions from the original baseline. Each arm has one score only; the outcomes cannot estimate a stable mean or prove that either change is harmful.

In [5]:
for r in followups:
    print(f"{r['experiment']}: {r['change']} {r['baseline_value']} -> {r['candidate_value']}; score={r['observed_score']}; repeats={r['repeats']}")


I02: max_output_tokens 8192 -> 16384; score=0.12; repeats=1
B2: thinking_budget 4096 -> 2048; score=0.15; repeats=1


## 5. Demonstrate canonical payload hashing

Two ZIPs below contain the same member paths and bytes but have different ZIP timestamps. The raw archive hashes differ; the canonical payload fingerprints match. This is the identity rule used by the paper.

In [6]:
def canonical_payload_fingerprint(path):
    members = []
    with zipfile.ZipFile(path) as zf:
        seen = set()
        for info in zf.infolist():
            if info.is_dir():
                continue
            if info.filename in seen:
                raise ValueError(f'duplicate member path: {info.filename}')
            seen.add(info.filename)
            raw = zf.read(info)
            members.append({'path': info.filename, 'size': len(raw), 'sha256': hashlib.sha256(raw).hexdigest()})
    members.sort(key=lambda x: x['path'])
    canonical = json.dumps(members, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(canonical).hexdigest()

with tempfile.TemporaryDirectory() as d:
    p1, p2 = Path(d)/'one.zip', Path(d)/'two.zip'
    for path, date in [(p1,(2026,1,1,0,0,0)), (p2,(2026,2,1,0,0,0))]:
        with zipfile.ZipFile(path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            info = zipfile.ZipInfo('agent.yaml', date_time=date)
            zf.writestr(info, 'model: demo\ntools: [read_file]\n')
    raw1 = hashlib.sha256(p1.read_bytes()).hexdigest()
    raw2 = hashlib.sha256(p2.read_bytes()).hexdigest()
    fp1, fp2 = canonical_payload_fingerprint(p1), canonical_payload_fingerprint(p2)
    print('Raw archive hashes equal:', raw1 == raw2)
    print('Canonical payload hashes equal:', fp1 == fp2)
    assert raw1 != raw2 and fp1 == fp2


Raw archive hashes equal: False
Canonical payload hashes equal: True


## 6. Interpretation and limits

**What this audit supports:** the supplied 14 scored archives contain 12 distinct payloads; one baseline payload has three different observed scores; the two later one-factor trials scored below its best observed draw; and one adapter package had zero-valued LoRA B matrices, so its nominal low-rank update was zero.

**What it does not support:** a causal explanation for score variation, a stable expected score, a claim that 16,384 output tokens or 2,048 thinking tokens are generally harmful, or a prediction of private-leaderboard rank. The 58-task conversion is participant-derived. The factor matrix is observational and small. No per-task outcomes, hosted traces, or scorer deployment receipts were available.

For the full command-line archive inspector, tensor inspector, tests, evidence ledger, and data manifest, see the companion **ASFA Artifact Forensics Resource** ZIP. The notebook is self-contained so the paper's case-study summary can be rerun without external downloads.